Assembles silver_job_postings from both staging tables via MERGE INTO (Day 10).
Reads:  jobmarket.silver.stg_postings_api, stg_postings_kaggle
Writes: jobmarket.silver.silver_job_postings (the real Silver table)
Idempotent: rerunning produces zero new rows (proven in final cell).

In [0]:
from pyspark.sql import functions as F

api    = spark.table("jobmarket.silver.stg_postings_api")
kaggle = spark.table("jobmarket.silver.stg_postings_kaggle")

# Final Silver schema — each source SELECTs into the same shape.
# Staging scaffolding stays behind: semantic_key (posting_id encodes it),
# _raw salary columns (audit trail lives in staging), api_request_id.

silver_api = api.select(
    "posting_id",
    "source",
    "source_posting_id",
    "title_raw",
    "title_norm",
    "seniority",
    "company",
    "city",
    "state",
    "is_remote",
    "salary_min",
    "salary_max",
    "salary_is_estimated",        # Adzuna has this natively (Day 4 flag)
    "description",
    "posted_date",
    "ingest_ts",
)

silver_kaggle = kaggle.select(
    "posting_id",
    "source",
    "source_posting_id",
    "title_raw",
    "title_norm",
    "seniority",
    "company",
    "city",
    "state",
    "is_remote",
    "salary_min",
    "salary_max",
    # Kaggle salaries are employer-disclosed -> estimated = false...
    # but ONLY where a salary exists; no salary -> null, not false
    F.when(F.col("salary_min").isNotNull(), F.lit(False))
     .otherwise(F.lit(None).cast("boolean")).alias("salary_is_estimated"),
    "description",
    "posted_date",
    "ingest_ts",
)

incoming = silver_api.unionByName(silver_kaggle)

print(f"API rows:    {silver_api.count()}")
print(f"Kaggle rows: {silver_kaggle.count()}")
print(f"Union:       {incoming.count()}")
incoming.groupBy("source").count().display()

In [0]:
# Same posting_id from both sources = same job caught by both feeds.
# This is THE cross-board duplication number (measured before merging).

overlap = (incoming
    .groupBy("posting_id")
    .agg(F.countDistinct("source").alias("n_sources"),
         F.count("*").alias("n_rows"))
    .filter("n_sources > 1"))

n_overlap = overlap.count()
print(f"Jobs present in BOTH sources: {n_overlap}")

if n_overlap > 0:
    # eyeball them — same real job?
    (overlap.join(incoming, "posting_id")
        .select("posting_id", "source", "title_raw", "company",
                "city", "posted_date", "salary_min")
        .orderBy("posting_id", "source")
        .display())

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS jobmarket.silver.silver_job_postings (
  posting_id           STRING,
  source               STRING,
  source_posting_id    STRING,
  title_raw            STRING,
  title_norm           STRING,
  seniority            STRING,
  company              STRING,
  city                 STRING,
  state                STRING,
  is_remote            BOOLEAN,
  salary_min           DOUBLE,
  salary_max           DOUBLE,
  salary_is_estimated  BOOLEAN,
  description          STRING,
  posted_date          DATE,
  ingest_ts            TIMESTAMP
) USING DELTA
PARTITIONED BY (posted_date)
""")
print("Table ready.")

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "jobmarket.silver.silver_job_postings")

(target.alias("t")
    .merge(
        incoming.alias("s"),
        "t.posting_id = s.posting_id"          # the match condition — Day 9's hash earning its keep
    )
    .whenMatchedUpdateAll(
        # survivorship across runs/sources: incoming row wins ONLY if it
        # brings better salary info — real beats estimated, something beats nothing
        condition="""
            (s.salary_min IS NOT NULL AND t.salary_min IS NULL)
            OR (s.salary_is_estimated = false AND t.salary_is_estimated = true)
        """
    )
    .whenNotMatchedInsertAll()
    .execute())

print("MERGE complete.")
print("Silver rows:", spark.table("jobmarket.silver.silver_job_postings").count())

In [0]:
before = spark.table("jobmarket.silver.silver_job_postings").count()

(target.alias("t")
    .merge(incoming.alias("s"), "t.posting_id = s.posting_id")
    .whenMatchedUpdateAll(
        condition="""
            (s.salary_min IS NOT NULL AND t.salary_min IS NULL)
            OR (s.salary_is_estimated = false AND t.salary_is_estimated = true)
        """)
    .whenNotMatchedInsertAll()
    .execute())

after = spark.table("jobmarket.silver.silver_job_postings").count()

print(f"Before rerun: {before}")
print(f"After rerun:  {after}")
assert before == after, "MERGE is NOT idempotent — row count changed!"
print("✅ Idempotency proven: rerun produced zero new rows")